# AgriSmart AI - Model 2 Self-Contained Auto-Training Notebook (v2)
=========================================================================

**Field-Domain Adaptation Training Pipeline for EfficientNet-B2**

This notebook is completely self-contained for Google Colab T4 GPU runtimes.

### Quick Start:
1. Connect to a **GPU Runtime** (Runtime -> Change runtime type -> T4 GPU).
2. Click **Runtime -> Run all**.

---

### CELL 1 - ENVIRONMENT SETUP
Automatically detects GPU, installs required packages, requires GPU, and sets random seeds.

In [ ]:
# Cell 1: Environment, GPU Detection & Random Seeds
import os
import sys
import subprocess
import random
import numpy as np
import torch

# Safe UTF-8 output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("[CELL 1] Setting up environment and detecting GPU...")

# Install required packages quietly
reqs = ["timm", "huggingface_hub", "albumentations", "scikit-learn", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)

# Detect CUDA
if not torch.cuda.is_available():
    raise RuntimeError("[CRITICAL ERROR] CUDA is not available. Please switch runtime type to T4 GPU!")

gpu_name = torch.cuda.get_device_name(0)
print(f"[OK] CUDA GPU Detected: {gpu_name}")
print(f"     PyTorch Version:  {torch.__version__}")

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[OK] Global random seed set to {SEED}.")

### CELL 2 - CLONE REPOSITORY
Clones the AgriSmart-AI repository, checkouts main, and prints the current commit hash.

In [ ]:
# Cell 2: Clone Repository & Workspace Setup
from pathlib import Path

repo_url = "https://github.com/Parrthiv125/AgriSmart-AI.git"
target_dir = Path("/content/AgriSmart-AI")

if target_dir.exists():
    print(f"[OK] Repository directory exists at {target_dir}.")
    os.chdir(target_dir)
    subprocess.run(["git", "checkout", "main"], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    if Path("models/classes.json").exists():
        print(f"[OK] Currently inside AgriSmart-AI workspace: {os.getcwd()}")
    else:
        print(f"Cloning {repo_url} into {target_dir}...")
        subprocess.run(["git", "clone", repo_url, str(target_dir)], check=True)
        os.chdir(target_dir)

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"[OK] Working Directory: {os.getcwd()}")
print(f"[OK] Current Commit Hash: {commit_hash}")

### CELL 3 - MODEL 1 CHECKPOINT VERIFICATION
Verifies Model 1 integrity (`models/agrismart_best.pth`). Model 1 MUST remain untouched.

In [ ]:
# Cell 3: Model 1 Checkpoint Verification
import hashlib
import json
import torch

model1_path = Path("models/agrismart_best.pth")
expected_hash = "af9684b036dac12c650004a2877add20708007ad34811015d1efaf5bcea06215"

if not model1_path.exists():
    raise FileNotFoundError(f"[CRITICAL ERROR] Model 1 checkpoint missing at {model1_path}!")

current_hash = hashlib.sha256(model1_path.read_bytes()).hexdigest()
if current_hash != expected_hash:
    raise ValueError(f"[CRITICAL ERROR] Model 1 SHA256 mismatch! Expected {expected_hash}, got {current_hash}")

print(f"[OK] Model 1 SHA256 verified: {current_hash[:16]}... (UNTOUCHED)")

ckpt = torch.load(model1_path, map_location="cpu", weights_only=False)
print(f"[OK] Model 1 Architecture: {ckpt.get('model_name', 'efficientnet_b2')}")
print(f"[OK] Model 1 Classes:      {ckpt.get('num_classes', 28)}")
print(f"[OK] Model 1 Input Res:    {ckpt.get('image_size', 260)}x{ckpt.get('image_size', 260)}")
print(f"[OK] Model 1 Best Epoch:   {ckpt.get('epoch')}")
print(f"[OK] Model 1 Val F1:       {ckpt.get('val_macro_f1', 0.0):.4f}")

### CELL 4 - DOWNLOAD PLANTVILLAGE DATASET
Downloads PlantVillage dataset using Hugging Face `data.zip` with fast system `unzip` extraction.

In [ ]:
# Cell 4: Idempotent PlantVillage Download & Fast Extraction
from data.download_dataset import download_and_extract_plantvillage

print("[CELL 4] Downloading & preparing PlantVillage dataset...")
raw_color_dir = download_and_extract_plantvillage()
print(f"[OK] PlantVillage RGB data ready at: {raw_color_dir}")

### CELL 5 - PLANTVILLAGE 28-CLASS SPLIT
Splits PlantVillage into 80/10/10 train/val/test using canonical 28-class mapping.

In [ ]:
# Cell 5: Execute PlantVillage 28-Class Split
from data.split_dataset import split_dataset

print("[CELL 5] Executing PlantVillage 28-class split (80/10/10, seed 42)...")
stats = split_dataset()
print(f"[OK] PlantVillage Split Complete: Train={stats['splits']['train']['total']} | Val={stats['splits']['val']['total']} | Test={stats['splits']['test']['total']}")

### CELL 6 - DOWNLOAD PLANTDOC DATASET
Downloads the public PlantDoc dataset and organizes train / test splits.

In [ ]:
# Cell 6: Download & Prepare PlantDoc Dataset
from data.download_plantdoc import download_plantdoc
from data.prepare_plantdoc import prepare_plantdoc_dataset

print("[CELL 6] Downloading PlantDoc raw dataset...")
download_plantdoc()

print("Organizing PlantDoc train & test sets...")
pd_stats = prepare_plantdoc_dataset()
print(f"[OK] PlantDoc Ready: Train={pd_stats['counts']['train']['total']} | LOCKED Test={pd_stats['counts']['test']['total']}")

### CELL 7 - SAFE PLANTDOC TRAIN/VALIDATION SPLIT & LEAKAGE CHECK
Splits PlantDoc TRAIN into 80% train and 20% validation. Performs SHA256 content-hash deduplication against locked test sets.

In [ ]:
# Cell 7: Safe PlantDoc 80/20 Split & SHA256 Content-Hash Deduplication
from data.prepare_model2_dataset import prepare_model2_dataset

print("[CELL 7] Building combined Model 2 dataset with SHA256 content deduplication...")
model2_stats = prepare_model2_dataset()
print(f"[OK] Content-Hash Leakage Check Passed: {model2_stats['leakage_passed']}")
print(f"[OK] Excluded Duplicate Content Count: {model2_stats['excluded_duplicates_count']}")

### CELL 8 - BUILD MODEL 2 DATASET SUMMARY
Prints comprehensive summary of the combined Model 2 dataset (`data/processed_model2`).

In [ ]:
# Cell 8: Dataset Summary & Class Counts
import json
with open("data/model2_split_stats.json", "r", encoding="utf-8") as f:
    info = json.load(f)

print("=" * 70)
print(" MODEL 2 DATASET COMPOSITION SUMMARY")
print("=" * 70)
print(f"  Combined Train Total:        {info['counts']['train']['total']:>6}")
print(f"    - PlantVillage Train:     {info['counts']['train']['plantvillage_total']:>6}")
print(f"    - PlantDoc Train Subset:  {info['counts']['train']['plantdoc_total']:>6}")
print(f"  Combined Val Total:          {info['counts']['val']['total']:>6}")
print(f"    - PlantVillage Val:       {info['counts']['val']['plantvillage_total']:>6}")
print(f"    - PlantDoc Val Subset:    {info['counts']['val']['plantdoc_total']:>6}")
print(f"  Excluded Duplicate Images:   {info['excluded_duplicates_count']:>6}")
print(f"  LOCKED PlantDoc TEST Set:    {233:>6} (UNTOUCHED)")
print(f"  LOCKED PlantVillage TEST:    {3835:>6} (UNTOUCHED)")
print("=" * 70)

### CELL 9 - FIELD DATA BALANCING
Configures `WeightedRandomSampler` with `plantdoc_oversample_factor = 5.0`.

In [ ]:
# Cell 9: Field Data Sampling Strategy
plantdoc_oversample_factor = 5.0

print(f"[OK] Configured PlantDoc Oversample Factor: {plantdoc_oversample_factor}x")
print("     Method: PyTorch WeightedRandomSampler (Zero physical duplication)")
print("     Weighting: PlantVillage samples (w=1.0) | PlantDoc samples (w=5.0)")

### CELL 10 - PIPELINE & GUARDRAIL VERIFICATION
Executes comprehensive verification checks before starting full training.

In [ ]:
# Cell 10: Model 2 Pipeline Verification
from data.verify_model2_pipeline import verify_model2_pipeline

print("[CELL 10] Running pre-training verification script...")
success = verify_model2_pipeline()
if not success:
    raise RuntimeError("[CRITICAL ERROR] Pipeline verification failed! Stopping notebook before training.")
print("[OK] All pre-training verifications & guardrails passed!")

### CELL 11 - DRY RUN
Executes a fast 1-epoch dry run to test optimizer, loss, AMP, and checkpoint writing.

In [ ]:
# Cell 11: Execute Dry-Run Verification
from training.train_model2 import run_training_model2

print("[CELL 11] Executing pipeline dry-run...")
dry_run_stats = run_training_model2(is_dry_run=True, plantdoc_oversample_factor=plantdoc_oversample_factor)
print("[OK] Dry run completed cleanly.")

### CELL 12 & 13 - MODEL 2 FULL TRAINING & AUTOMATIC RESUME SUPPORT
Trains Model 2 for 25 epochs. Automatically resumes from `models/agrismart_field_adapted_last.pth` if present.

In [ ]:
# Cell 12 & 13: Full Model 2 Training & Automatic Resume Support
print("[CELL 12 & 13] Launching Model 2 Full Field-Domain Adaptation Training (25 Epochs)...")
print("        - Model 1 Checkpoint: models/agrismart_best.pth (UNTOUCHED)")
print("        - PlantDoc TEST Set:  data/plantdoc/test (LOCKED & UNTOUCHED)")
print(f"        - PlantDoc Oversample Factor: {plantdoc_oversample_factor}x")
print()

final_metadata = run_training_model2(
    epochs=25,
    batch_size=32,
    lr=1e-4,
    plantdoc_oversample_factor=plantdoc_oversample_factor,
    is_dry_run=False,
)

### CELL 14 - FINAL TRAINING SUMMARY & GUARDRAIL CONFIRMATION
Summarizes training results and verifies that all locked test sets remained untouched.

In [ ]:
# Cell 14: Final Training Summary & Guardrail Audit
import hashlib
import json
from pathlib import Path

best_ckpt = Path("models/agrismart_field_adapted_best.pth")
last_ckpt = Path("models/agrismart_field_adapted_last.pth")
metadata_file = Path("models/agrismart_field_adapted_metadata.json")
model1_file = Path("models/agrismart_best.pth")

print("=" * 70)
print(" FINAL MODEL 2 TRAINING SUMMARY & GUARDRAIL AUDIT")
print("=" * 70)

if metadata_file.exists():
    with open(metadata_file, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print(f"  Best Epoch:                  {meta['training_results']['best_epoch']}")
    print(f"  Best Validation Macro-F1:    {meta['training_results']['best_val_macro_f1']:.4f}")
    print(f"  Sampling Method:             {meta['sampling_strategy']['method']}")
    print(f"  PlantDoc Oversample Factor:  {meta['sampling_strategy']['plantdoc_oversample_factor']}x")

print(f"  Best Checkpoint Saved:       {best_ckpt} ({best_ckpt.stat().st_size / 1e6:.1f} MB)")
print(f"  Last Checkpoint Saved:       {last_ckpt} ({last_ckpt.stat().st_size / 1e6:.1f} MB)")

# Verify Model 1 SHA256 integrity
m1_hash = hashlib.sha256(model1_file.read_bytes()).hexdigest()
expected_m1_hash = "af9684b036dac12c650004a2877add20708007ad34811015d1efaf5bcea06215"
print(f"  Model 1 SHA256 Integrity:    {'VERIFIED UNTOUCHED' if m1_hash == expected_m1_hash else 'FAILED Altered!'}")

# Confirm Locked Test Sets
pd_test_count = len(list(Path("data/plantdoc/test").rglob("*")))
pv_test_count = len(list(Path("data/processed/test").rglob("*")))
print(f"  PlantDoc TEST Status:        LOCKED & UNTOUCHED ({pd_test_count} items)")
print(f"  PlantVillage TEST Status:     LOCKED & UNTOUCHED ({pv_test_count} items)")
print(f"  PlantDoc TEST Evaluated:     FALSE (Evaluation locked until model frozen)")
print("=" * 70)
print(" [SUCCESS] MODEL 2 TRAINING PIPELINE FINISHED SUCCESSFULLY.")